In [ ]:
# Build analysis panel
#
# Inputs
#   ../processed/cmc_ohlcv_combined.csv    – combined OHLCV, Feb 2024 – Mar 2026
#   ../processed/tord_first_offering.csv   – one row per token, first offering only
#   ../processed/cmc_token_metadata.csv    – CMC token metadata
#
# Key design choices
#   - Entry gate     : drop token-days before ico_start (no look-ahead on offering info)
#   - Activity gate  : token enters sample only after 50 cumulative active days
#   - Rolling filter : retain only days where token had >= 15 non-zero-volume days
#                      in the trailing 30 calendar days
#   - Liquidity      : raw daily measures smoothed to trailing 30-day rolling median
#                      before stress classification and use as predictive features
#   - Stress label   : asset-specific expanding-window 90th percentile of
#                      rolling-median Amihud (no look-ahead bias)
#   - Crash label    : asset-specific expanding-window 10th percentile of
#                      daily returns (no look-ahead bias)
#   - Market factor  : within-asset z-score via expanding mean/std (no look-ahead bias)
#
# Output
#   ../processed/analysis_panel.csv

from pathlib import Path
import numpy as np
import pandas as pd

# ===== CONFIG =====
PROCESSED = Path("../processed")

OHLCV_COMBINED = PROCESSED / "cmc_ohlcv_combined.csv"
TORD_FIRST     = PROCESSED / "tord_first_offering.csv"
CMC_META       = PROCESSED / "cmc_token_metadata.csv"

OUT_PANEL = PROCESSED / "analysis_panel.csv"

# Sample filter thresholds
ENTRY_ACTIVE_DAYS   = 50   # cumulative active days before token enters sample
ROLLING_ACTIVE_DAYS = 15   # min active days in trailing 30-day window
ROLLING_WINDOW      = 30   # calendar days for rolling activity check
MIN_OBS_LABEL       = 60   # min qualifying obs before expanding quantile becomes valid
ROLL_LIQ            = 30   # trailing days for rolling-median liquidity smoothing
ROLL_VOL            = 20   # trailing days for rolling volatility

In [2]:
# ── 1. Load and merge ─────────────────────────────────────────────────────────

ohlcv = pd.read_csv(OHLCV_COMBINED, low_memory=False)
ohlcv["date"]   = pd.to_datetime(ohlcv["date"], errors="coerce")
ohlcv["cmc_id"] = pd.to_numeric(ohlcv["cmc_id"], errors="coerce").astype("Int64")
ohlcv = ohlcv.dropna(subset=["cmc_id", "date"]).copy()

tord = pd.read_csv(TORD_FIRST, low_memory=False)
tord["cmc_id"]    = pd.to_numeric(tord["cmc_id"], errors="coerce").astype("Int64")
tord["ico_start"] = pd.to_datetime(tord["ico_start"], errors="coerce")
tord = tord.dropna(subset=["cmc_id"]).copy()

meta = pd.read_csv(CMC_META, low_memory=False)
meta["cmc_id"] = pd.to_numeric(meta["cmc_id"], errors="coerce").astype("Int64")
meta = meta.dropna(subset=["cmc_id"]).drop_duplicates("cmc_id").copy()

tord_cols = [
    "cmc_id", "ico_start", "ico_end", "is_ico", "is_ieo", "is_sto",
    "price_usd", "raised_usd", "distributed_in_ico_frac",
    "whitelist", "kyc", "bonus", "restricted_areas",
    "teamsize", "rating", "ERC20",
    "has_whitepaper", "has_github", "has_linkedin", "has_website",
    "ico_duration_days", "has_presale",
    "flag_suspicious_ico_start", "flag_negative_price", "flag_negative_raised",
    "min_inv_usd_value",
]
tord_cols = [c for c in tord_cols if c in tord.columns]

meta_cols = ["cmc_id", "category", "platform_name", "date_added", "tags"]
meta_cols = [c for c in meta_cols if c in meta.columns]

panel = (
    ohlcv
    .merge(tord[tord_cols], on="cmc_id", how="left")
    .merge(meta[meta_cols], on="cmc_id", how="left")
)

print(f"After merge : {len(panel):>10,} rows | {panel['cmc_id'].nunique()} tokens")
print(f"Date range  : {panel['date'].min().date()} → {panel['date'].max().date()}")

After merge :  1,076,915 rows | 1853 tokens
Date range  : 2024-02-13 → 2026-03-30


In [3]:
# ── 2. Entry gate: drop token-days before ico_start ───────────────────────────

panel["flag_no_ico_start"] = panel["ico_start"].isna().astype(int)

has_start   = panel["ico_start"].notna()
before_gate = has_start & (panel["date"] < panel["ico_start"])

n_before = len(panel)
panel = panel[~before_gate].copy()

print(f"Rows dropped by ico_start entry gate : {n_before - len(panel):>8,}")
print(f"Rows remaining                       : {len(panel):>8,} | {panel['cmc_id'].nunique()} tokens")

# ── Restrict to TORD-matched tokens only ──────────────────────────────────────
n_before_tord = len(panel)
panel = panel[panel["flag_no_ico_start"] == 0].copy()
print(f"Rows dropped (no TORD match)         : {n_before_tord - len(panel):>8,}")
print(f"Rows remaining (TORD-matched only)   : {len(panel):>8,} | {panel['cmc_id'].nunique()} tokens")

Rows dropped by ico_start entry gate :        0
Rows remaining                       : 1,076,915 | 1853 tokens
Rows dropped (no TORD match)         :  757,741
Rows remaining (TORD-matched only)   :  319,174 | 531 tokens


In [4]:
# ── 3. Activity-based sample filters ─────────────────────────────────────────
# Uses date-indexed groupby rolling rather than groupby-apply to avoid
# pandas 3.x key-column stripping behaviour.

panel = panel.sort_values(["cmc_id", "date"]).reset_index(drop=True)
panel["active_day"] = (pd.to_numeric(panel["volume"], errors="coerce").fillna(0) > 0).astype(int)

# 3a. Entry gate: cumulative active days per token
panel["cum_active"] = panel.groupby("cmc_id")["active_day"].cumsum()
panel["pass_entry"] = (panel["cum_active"] >= ENTRY_ACTIVE_DAYS).astype(int)

# 3b. Rolling activity filter: active days in trailing ROLLING_WINDOW calendar days
# Set date as index so pandas uses calendar-day windows, then merge result back.
_tmp = panel[["cmc_id", "date", "active_day"]].set_index("date")
_roll = (
    _tmp.groupby("cmc_id")["active_day"]
        .rolling(f"{ROLLING_WINDOW}D", min_periods=1)
        .sum()
        .reset_index()                                   # → cmc_id, date, active_day
        .rename(columns={"active_day": "rolling_active_30d"})
        .sort_values(["cmc_id", "date"])
)
# shift(1) within token so the current day is excluded from its own window
_roll["rolling_active_30d"] = (
    _roll.groupby("cmc_id")["rolling_active_30d"].shift(1).fillna(0)
)
panel = panel.merge(_roll[["cmc_id", "date", "rolling_active_30d"]], on=["cmc_id", "date"], how="left")

panel["pass_rolling"] = (panel["rolling_active_30d"] >= ROLLING_ACTIVE_DAYS).astype(int)
panel["in_sample"]    = ((panel["pass_entry"] == 1) & (panel["pass_rolling"] == 1)).astype(int)

n_total  = len(panel)
n_sample = panel["in_sample"].sum()
n_tokens = panel.loc[panel["in_sample"] == 1, "cmc_id"].nunique()

print(f"Total rows (post ico_start gate)  : {n_total:>8,}")
print(f"Rows passing entry gate only      : {panel['pass_entry'].sum():>8,}")
print(f"Rows passing rolling filter only  : {panel['pass_rolling'].sum():>8,}")
print(f"Rows passing BOTH filters         : {n_sample:>8,} | {n_tokens} tokens")

Total rows (post ico_start gate)  :  319,174
Rows passing entry gate only      :  254,577
Rows passing rolling filter only  :  245,981
Rows passing BOTH filters         :  231,354 | 420 tokens


In [5]:
# ── 4. Daily returns and raw liquidity measures ───────────────────────────────

for col in ["open", "high", "low", "close", "volume", "market_cap"]:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

panel["ret"] = panel.groupby("cmc_id")["close"].pct_change()

panel["amihud_raw"] = np.where(
    panel["volume"] > 0,
    panel["ret"].abs() / panel["volume"],
    np.nan
)

panel["turnover_raw"] = np.where(
    panel["market_cap"] > 0,
    panel["volume"] / panel["market_cap"],
    np.nan
)

denom = panel["high"] + panel["low"]
panel["spread_raw"] = np.where(
    denom > 0,
    2 * (panel["high"] - panel["low"]) / denom,
    np.nan
)

panel["log_mcap"] = np.log1p(panel["market_cap"].clip(lower=0))

# ── A: Corwin-Schultz (2012) proportional spread estimator ────────────────────
# Uses OHLC data from consecutive day pair (t-1, t).
# beta  = sum of squared log HL ranges over 2 days
# gamma = squared log range of the 2-day high/low envelope
# alpha = derived price-impact coefficient; spread = 2*(e^alpha-1)/(1+e^alpha)
# Negative alpha (trending markets) gives negative spread; clipped to 0 per CS paper.
_EPS      = 1e-8
_high_tm1 = panel.groupby("cmc_id")["high"].shift(1).clip(lower=_EPS)
_low_tm1  = panel.groupby("cmc_id")["low"].shift(1).clip(lower=_EPS)
_h_t      = np.log(panel["high"].clip(lower=_EPS) / panel["low"].clip(lower=_EPS))
_h_tm1    = np.log(_high_tm1 / _low_tm1)
_hi_2d    = pd.concat([panel["high"], panel.groupby("cmc_id")["high"].shift(1)],
                       axis=1).max(axis=1)
_lo_2d    = pd.concat([panel["low"],  panel.groupby("cmc_id")["low"].shift(1)],
                       axis=1).min(axis=1)

_beta  = _h_t**2 + _h_tm1**2
_gamma = np.log(_hi_2d.clip(lower=_EPS) / _lo_2d.clip(lower=_EPS))**2
_k     = 3 - 2 * np.sqrt(2)          # constant ≈ 0.1716
_alpha = np.clip(
    (np.sqrt(2 * _beta) - np.sqrt(_beta)) / _k - np.sqrt(_gamma / _k),
    -50, 50                           # guard against overflow in exp()
)
_cs_val = 2 * (np.exp(_alpha) - 1) / (1 + np.exp(_alpha))

panel["cs_spread_raw"] = np.where(
    panel["high"].notna() & panel["low"].notna()
    & panel.groupby("cmc_id")["high"].shift(1).notna(),
    _cs_val.clip(lower=0, upper=1.0),
    np.nan
)

# ── C: Amivest liquidity ratio ─────────────────────────────────────────────────
# Amivest = Volume / |Return|  (higher = more liquid; inverse scale to Amihud).
panel["amivest_raw"] = np.where(
    (panel["volume"] > 0) & (panel["ret"].abs() > _EPS),
    panel["volume"] / panel["ret"].abs(),
    np.nan
)

# ── C: Pastor-Stambaugh signed order flow ─────────────────────────────────────
# signed_flow = sign(ret) * turnover.  Used in cell-06 to compute rolling PS gamma.
panel["ps_sf_raw"] = np.where(
    panel["ret"].notna() & panel["turnover_raw"].notna(),
    np.sign(panel["ret"]) * panel["turnover_raw"],
    np.nan
)

print("Raw liquidity coverage (non-NaN share):")
for col in ["ret", "amihud_raw", "turnover_raw", "spread_raw",
            "cs_spread_raw", "amivest_raw", "ps_sf_raw"]:
    pct = panel[col].notna().mean() * 100
    print(f"  {col:<20}: {pct:.1f}%")


Raw liquidity coverage (non-NaN share):
  ret                 : 99.8%
  amihud_raw          : 78.7%
  turnover_raw        : 65.8%
  spread_raw          : 100.0%
  cs_spread_raw       : 99.8%
  amivest_raw         : 78.5%
  ps_sf_raw           : 65.7%


In [ ]:
# ── 5. Trailing rolling median liquidity + PS gamma ───────────────────────────
# Same date-indexed groupby rolling pattern.
# min_periods=15 avoids producing values from very sparse early windows.

_base = panel[["cmc_id", "date",
               "amihud_raw", "turnover_raw", "spread_raw",
               "cs_spread_raw", "amivest_raw", "ret"]].set_index("date")

smooth_parts = []
for raw, smooth, agg in [
    ("amihud_raw",    "amihud_med30",   "median"),
    ("turnover_raw",  "turnover_med30", "median"),
    ("spread_raw",    "spread_med30",   "median"),
    ("cs_spread_raw", "cs_spread_30d",  "median"),
    ("amivest_raw",   "amivest_30d",    "median"),
    ("ret",           "volatility_20d", "std"),
]:
    win    = f"{ROLL_LIQ}D" if raw != "ret" else f"{ROLL_VOL}D"
    minp   = 15             if raw != "ret" else 10
    roller = _base.groupby("cmc_id")[raw].rolling(win, min_periods=minp)
    s = getattr(roller, agg)().reset_index().rename(columns={raw: smooth})
    smooth_parts.append(s.set_index(["cmc_id", "date"])[smooth])

smooth_df = pd.concat(smooth_parts, axis=1).reset_index()
panel = panel.merge(smooth_df, on=["cmc_id", "date"], how="left")

# ── C: Rolling PS gamma — per-asset OLS of ret_t ~ sf_lag_{t-1} ───────────────
# gamma_t = slope(ret ~ signed_flow_lag) over trailing ROLL_LIQ-day window.
# Estimated via rolling means: beta = (E[xy]-E[x]E[y]) / (E[x^2]-E[x]^2).
# More negative gamma => stronger return reversal => less liquid (PS 2003).
_ps = (panel[["cmc_id", "date", "ret", "ps_sf_raw"]]
       .sort_values(["cmc_id", "date"])
       .copy())
_ps["sf_lag"] = _ps.groupby("cmc_id")["ps_sf_raw"].shift(1)
_ps["xy"]     = _ps["ret"] * _ps["sf_lag"]
_ps["y2"]     = _ps["sf_lag"] ** 2
_ps_idx = _ps.set_index("date")

for _col in ["ret", "sf_lag", "xy", "y2"]:
    _ps_idx[f"rm_{_col}"] = (
        _ps_idx.groupby("cmc_id")[_col]
               .rolling(f"{ROLL_LIQ}D", min_periods=15)
               .mean()
               .reset_index(level=0, drop=True)
    )

_denom = _ps_idx["rm_y2"] - _ps_idx["rm_sf_lag"] ** 2
_ps_idx["ps_gamma"] = np.where(
    _denom.abs() > 1e-14,
    (_ps_idx["rm_xy"] - _ps_idx["rm_ret"] * _ps_idx["rm_sf_lag"]) / _denom,
    np.nan
)
panel = panel.merge(
    _ps_idx.reset_index()[["cmc_id", "date", "ps_gamma"]],
    on=["cmc_id", "date"], how="left"
)

print("Smoothed liquidity coverage (non-NaN share, full panel):")
for col in ["amihud_med30", "turnover_med30", "spread_med30",
            "cs_spread_30d", "amivest_30d", "volatility_20d", "ps_gamma"]:
    pct = panel[col].notna().mean() * 100
    print(f"  {col:<20}: {pct:.1f}%")


Smoothed liquidity coverage (non-NaN share, full panel):
  amihud_med30        : 77.0%
  turnover_med30      : 64.4%
  spread_med30        : 97.6%
  cs_spread_30d       : 97.5%
  amivest_30d         : 77.0%
  volatility_20d      : 98.3%
  ps_gamma            : 60.8%


In [7]:
# ── 6. Winsorise at 1st / 99th percentile (in-sample rows only) ──────────────

WINSOR_COLS = [
    "amihud_med30", "turnover_med30", "spread_med30",
    "cs_spread_30d", "amivest_30d", "ps_gamma",
    "amihud_raw", "turnover_raw", "spread_raw",
    "cs_spread_raw", "amivest_raw",
    "volatility_20d", "ret",
]

in_samp = panel["in_sample"] == 1

for col in WINSOR_COLS:
    if col not in panel.columns:
        continue
    lo = panel.loc[in_samp, col].quantile(0.01)
    hi = panel.loc[in_samp, col].quantile(0.99)
    panel[col] = panel[col].clip(lower=lo, upper=hi)

print("Winsorisation complete (1st/99th percentile, in-sample quantiles).")
print(f"  New columns winsorised: cs_spread_30d, amivest_30d, ps_gamma")


Winsorisation complete (1st/99th percentile, in-sample quantiles).
  New columns winsorised: cs_spread_30d, amivest_30d, ps_gamma


In [8]:
# ── 7. Stress labels (rolling-median Amihud, asset-specific 90th pct) ─────────
# Expanding-window quantile: threshold at date t uses data up to t-1 only (no look-ahead).
# shift(1) within each token ensures the boundary was computed from past observations.

in_s = panel["in_sample"] == 1

# --- amihud expanding 90th pct ---
_stress_base = (
    panel.loc[in_s & panel["amihud_med30"].notna(), ["cmc_id", "date", "amihud_med30"]]
         .sort_values(["cmc_id", "date"])
         .copy()
)
_stress_base["_q90_amihud"] = (
    _stress_base.groupby("cmc_id")["amihud_med30"]
        .transform(lambda x: x.expanding(min_periods=MIN_OBS_LABEL).quantile(0.90))
)
_stress_base["_q90_amihud"] = (
    _stress_base.groupby("cmc_id")["_q90_amihud"].shift(1)
)

# --- turnover expanding 10th pct (for strict stress) ---
_turn_base = (
    panel.loc[in_s & panel["turnover_med30"].notna(), ["cmc_id", "date", "turnover_med30"]]
         .sort_values(["cmc_id", "date"])
         .copy()
)
_turn_base["_q10_turn"] = (
    _turn_base.groupby("cmc_id")["turnover_med30"]
        .transform(lambda x: x.expanding(min_periods=MIN_OBS_LABEL).quantile(0.10))
)
_turn_base["_q10_turn"] = (
    _turn_base.groupby("cmc_id")["_q10_turn"].shift(1)
)

panel = (
    panel
    .merge(_stress_base[["cmc_id", "date", "_q90_amihud"]], on=["cmc_id", "date"], how="left")
    .merge(_turn_base[["cmc_id", "date", "_q10_turn"]],     on=["cmc_id", "date"], how="left")
)

panel["stress_amihud"] = np.where(
    in_s & panel["amihud_med30"].notna() & panel["_q90_amihud"].notna()
    & (panel["_q90_amihud"] > 0),
    (panel["amihud_med30"] >= panel["_q90_amihud"]).astype(float),
    np.nan

)
panel["stress_amihud_strict"] = np.where(
    in_s & panel["amihud_med30"].notna() & panel["turnover_med30"].notna()
    & panel["_q90_amihud"].notna() & panel["_q10_turn"].notna() & (panel["_q90_amihud"] > 0),
    ((panel["amihud_med30"] >= panel["_q90_amihud"]) &
     (panel["turnover_med30"] <= panel["_q10_turn"])).astype(float),
    np.nan
)
panel = panel.drop(columns=["_q90_amihud", "_q10_turn"])

print("Stress label prevalence (in-sample rows with valid label):")
print(f"  stress_amihud        : {panel.loc[in_s, 'stress_amihud'].mean():.3f}  "
      f"(n={panel.loc[in_s, 'stress_amihud'].notna().sum():,})")
print(f"  stress_amihud_strict : {panel.loc[in_s, 'stress_amihud_strict'].mean():.3f}  "
      f"(n={panel.loc[in_s, 'stress_amihud_strict'].notna().sum():,})")

Stress label prevalence (in-sample rows with valid label):
  stress_amihud        : 0.274  (n=206,475)
  stress_amihud_strict : 0.116  (n=158,343)


In [9]:
# ── 8. Crash labels (asset-specific 10th percentile of daily returns) ─────────
# Expanding-window quantile: threshold at date t uses data up to t-1 only (no look-ahead).

_crash_base = (
    panel.loc[in_s & panel["ret"].notna(), ["cmc_id", "date", "ret"]]
         .sort_values(["cmc_id", "date"])
         .copy()
)
_crash_base["_q10_ret"] = (
    _crash_base.groupby("cmc_id")["ret"]
        .transform(lambda x: x.expanding(min_periods=MIN_OBS_LABEL).quantile(0.10))
)
_crash_base["_q10_ret"] = _crash_base.groupby("cmc_id")["_q10_ret"].shift(1)

panel = panel.merge(_crash_base[["cmc_id", "date", "_q10_ret"]], on=["cmc_id", "date"], how="left")
panel["crash"] = np.where(
    in_s & panel["ret"].notna() & panel["_q10_ret"].notna(),
    (panel["ret"] <= panel["_q10_ret"]).astype(float),
    np.nan
)
panel = panel.drop(columns=["_q10_ret"])

print(f"Crash label prevalence (in-sample): "
      f"{panel.loc[in_s, 'crash'].mean():.3f}  "
      f"(n={panel.loc[in_s, 'crash'].notna().sum():,})")

Crash label prevalence (in-sample): 0.106  (n=206,679)


In [10]:
# ── 9. Market-wide liquidity factors ─────────────────────────────────────────
# 9a: Amihud factor (cross-sectional mean of within-asset z-scored amihud_med30).
#     Retained for PanelOLS regression (Table 5) and backward compatibility.
# 9b: PCA factor (F) — PC1 across 4 standardised illiquidity proxies.
#     Higher PC1 = more market-wide illiquidity.

# ── 9a: Amihud z-score factor ────────────────────────────────────────────────
_z_base = (
    panel.loc[in_s & panel["amihud_med30"].notna(), ["cmc_id", "date", "amihud_med30"]]
         .sort_values(["cmc_id", "date"])
         .copy()
)
_z_base["_mu"] = (
    _z_base.groupby("cmc_id")["amihud_med30"]
        .transform(lambda x: x.expanding(min_periods=10).mean())
)
_z_base["_sd"] = (
    _z_base.groupby("cmc_id")["amihud_med30"]
        .transform(lambda x: x.expanding(min_periods=10).std())
)
_z_base["_mu"] = _z_base.groupby("cmc_id")["_mu"].shift(1)
_z_base["_sd"] = _z_base.groupby("cmc_id")["_sd"].shift(1)

panel = panel.merge(_z_base[["cmc_id", "date", "_mu", "_sd"]], on=["cmc_id", "date"], how="left")
panel["amihud_z"] = np.where(
    in_s & panel["amihud_med30"].notna() & panel["_sd"].notna() & (panel["_sd"] > 0),
    (panel["amihud_med30"] - panel["_mu"]) / panel["_sd"],
    np.nan
)
panel = panel.drop(columns=["_mu", "_sd"])

# Winsorise amihud_z before aggregating: near-zero volume days can produce
# z-scores of thousands, dominating the cross-sectional mean. Clip at 99.9th pct.
_z_hi  = panel.loc[in_s & panel["amihud_z"].notna(), "amihud_z"].quantile(0.999)
_z_lo  = panel.loc[in_s & panel["amihud_z"].notna(), "amihud_z"].quantile(0.001)
_n_clip = ((panel["amihud_z"] > _z_hi) | (panel["amihud_z"] < _z_lo)).sum()
panel["amihud_z"] = panel["amihud_z"].clip(lower=_z_lo, upper=_z_hi)
print(f"amihud_z winsorised at [0.1%, 99.9%]: [{_z_lo:.4f}, {_z_hi:.4f}]  "
      f"({_n_clip} obs clipped)")

mkt_factor = (
    panel.loc[in_s]
         .groupby("date")["amihud_z"]
         .mean()
         .rename("mkt_illiq_factor")
         .reset_index()
)
panel = panel.merge(mkt_factor, on="date", how="left")
print(f"Amihud z-score factor : {len(mkt_factor)} daily obs  "
      f"Mean={mkt_factor['mkt_illiq_factor'].mean():.4f}  "
      f"Max={mkt_factor['mkt_illiq_factor'].max():.4f}  "
      f"Std={mkt_factor['mkt_illiq_factor'].std():.4f}")

# ── 9b: F — PCA factor across 4 liquidity proxies ─────────────────────────────
# Proxies are aligned to "higher = more illiquid" before PCA:
#   amihud_med30, cs_spread_30d  : already higher = more illiquid
#   turnover_med30, amivest_30d  : inverted (1/x) so higher inverse = more illiquid
# PCA estimated on in-sample rows complete across all 4 proxies.
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.decomposition import PCA as _PCA

_pca_src_cols = ["amihud_med30", "cs_spread_30d", "turnover_med30", "amivest_30d"]
_pca_data     = panel.loc[in_s, ["cmc_id", "date"] + _pca_src_cols].copy()

_pca_data["_inv_turn"]   = 1.0 / _pca_data["turnover_med30"].replace(0, np.nan)
_pca_data["_inv_amivest"] = 1.0 / _pca_data["amivest_30d"].replace(0, np.nan)
_pca_mat_cols = ["amihud_med30", "cs_spread_30d", "_inv_turn", "_inv_amivest"]
_pca_ok       = _pca_data.dropna(subset=_pca_mat_cols).copy()

_n_in_s = int(in_s.sum())
print(f"PCA: {len(_pca_ok):,} in-sample rows complete across all 4 proxies "
      f"({100*len(_pca_ok)/_n_in_s:.1f}%)")

if len(_pca_ok) >= 200:
    _ss  = _SS()
    _X   = _ss.fit_transform(_pca_ok[_pca_mat_cols])
    _pca = _PCA(n_components=4)
    _pca.fit(_X)

    print("Explained variance ratio:")
    for _i, _ev in enumerate(_pca.explained_variance_ratio_):
        _cum = _pca.explained_variance_ratio_[:_i+1].sum()
        print(f"  PC{_i+1}: {_ev:.3f}  (cumul {_cum:.3f})")

    _loadings = {c: round(v, 3) for c, v in zip(_pca_mat_cols, _pca.components_[0])}
    print(f"PC1 loadings: {_loadings}")

    _pca_ok = _pca_ok.copy()
    _pca_ok["_pc1_raw"] = _pca.transform(_X)[:, 0]
    # Ensure PC1 is oriented so that higher = more illiquid (Amihud loads positively)
    if _pca.components_[0, 0] < 0:   # index 0 = amihud_med30
        _pca_ok["_pc1_raw"] = -_pca_ok["_pc1_raw"]

    mkt_pc1 = (
        _pca_ok.groupby("date")["_pc1_raw"]
               .mean()
               .rename("mkt_illiq_pc1")
               .reset_index()
    )
    panel = panel.merge(mkt_pc1, on="date", how="left")

    _corr_df = mkt_factor.merge(mkt_pc1, on="date")
    _r = _corr_df["mkt_illiq_factor"].corr(_corr_df["mkt_illiq_pc1"])
    print(f"PC1 factor: {len(mkt_pc1)} daily obs  "
          f"Corr(PC1, Amihud factor) = {_r:.3f}")
else:
    panel["mkt_illiq_pc1"] = np.nan
    print("Insufficient complete rows for PCA — mkt_illiq_pc1 set to NaN.")


amihud_z winsorised at [0.1%, 99.9%]: [-3.1116, 15.7587]  (452 obs clipped)
Amihud z-score factor : 727 daily obs  Mean=0.6781  Max=1.6711  Std=0.4304
PCA: 175,485 in-sample rows complete across all 4 proxies (75.9%)
Explained variance ratio:
  PC1: 0.586  (cumul 0.586)
  PC2: 0.252  (cumul 0.837)
  PC3: 0.162  (cumul 1.000)
  PC4: 0.000  (cumul 1.000)
PC1 loadings: {'amihud_med30': np.float64(0.631), 'cs_spread_30d': np.float64(0.037), '_inv_turn': np.float64(0.449), '_inv_amivest': np.float64(0.631)}
PC1 factor: 727 daily obs  Corr(PC1, Amihud factor) = 0.171


In [11]:
# ── 10. Coin indicator ────────────────────────────────────────────────────────

if "category" in panel.columns:
    panel["is_coin"] = (panel["category"].str.lower().str.strip() == "coin").astype("Int8")
else:
    panel["is_coin"] = pd.NA
    print("Warning: 'category' column not found in metadata.")

print("Asset type breakdown (unique tokens, in-sample):")
print(panel.loc[in_s]
          .drop_duplicates("cmc_id")["is_coin"]
          .value_counts(dropna=False)
          .rename({0: "token", 1: "coin"}))

Asset type breakdown (unique tokens, in-sample):
is_coin
token    337
coin      83
Name: count, dtype: Int64


In [12]:
# ── 11. Coverage summary ──────────────────────────────────────────────────────

final = panel[in_s].copy()
valid = final.dropna(subset=["ret", "amihud_med30", "stress_amihud"])

print("=" * 62)
print("ANALYSIS PANEL SUMMARY")
print("=" * 62)
print(f"Full panel rows        : {len(panel):>10,}")
print(f"In-sample rows         : {len(final):>10,}")
print(f"Valid (ret+liq+stress) : {len(valid):>10,}")
print(f"Tokens in-sample       : {final['cmc_id'].nunique():>10,}")
print(f"Tokens with valid obs  : {valid['cmc_id'].nunique():>10,}")
print(f"Date range (in-sample) : {final['date'].min().date()} to {final['date'].max().date()}")
print()
print("Liquidity proxy coverage (in-sample):")
for col in ["amihud_med30", "turnover_med30", "spread_med30",
            "cs_spread_30d", "amivest_30d", "ps_gamma",
            "mkt_illiq_factor", "mkt_illiq_pc1"]:
    if col in final.columns:
        pct = final[col].notna().mean() * 100
        print(f"  {col:<22}: {pct:.1f}%")
print("=" * 62)


ANALYSIS PANEL SUMMARY
Full panel rows        :    319,174
In-sample rows         :    231,354
Valid (ret+liq+stress) :    206,475
Tokens in-sample       :        420
Tokens with valid obs  :        403
Date range (in-sample) : 2024-04-02 to 2026-03-30

Liquidity proxy coverage (in-sample):
  amihud_med30          : 99.9%
  turnover_med30        : 76.0%
  spread_med30          : 100.0%
  cs_spread_30d         : 100.0%
  amivest_30d           : 99.8%
  ps_gamma              : 75.8%
  mkt_illiq_factor      : 98.8%
  mkt_illiq_pc1         : 100.0%


In [13]:
# ── 12. Save ──────────────────────────────────────────────────────────────────
# Full panel saved with all flags so robustness checks can vary thresholds
# without re-running this notebook.

panel.to_csv(OUT_PANEL, index=False)
print(f"Saved: {OUT_PANEL}")
print(f"Shape: {panel.shape}")
print(f"Size : {OUT_PANEL.stat().st_size / 1e6:.1f} MB")

Saved: ..\processed\analysis_panel.csv
Shape: (319174, 66)
Size : 211.6 MB
